# Initialize a dense polymer melt in one line

In [8]:
import warnings
import time
from flowermd.library import LJChain,RandomWalk,Bead_Spring_DPD, PhantomWalk
import unyt as u
import hoomd
import gsd.hoomd

warnings.filterwarnings("ignore")

In [17]:
def initialize_dense_cg_melt(
    number_molecules,
    chain_length,
    density
):
    molecules = LJChain(
        num_mols=number_molecules,
        lengths=chain_length,
        bead_sequence=["_A"],
        bead_mass={"_A": 1.0},
        bond_lengths={"_A-_A": 1.0},
        identify_connections=False,
    )
    
    ref_length = 1.0 * u.Unit("nm")
    ref_mass = 1.0 * u.Unit("g/mol")
    ref_energy = 1.0 * u.Unit("kcal / mol")
    ref_values_dict = {"length": ref_length, "mass": ref_mass, "energy": ref_energy}
    
    system = RandomWalk(
        molecules=molecules,
        density=density * u.Unit("nm**-3"),
        bond_length=1.0,
        buffer=0.5,
        base_units=ref_values_dict,
    )
    
    nlist = hoomd.md.nlist.Cell(buffer=0.1, exclusions=["bond"])
    system.apply_forcefield(
        force_field=Bead_Spring_DPD(),
        r_cut=1.01,
        kT=1.0,
        speedup_by_molgraph=True,
        speedup_by_moltag=False,
        nlist=nlist,
    )
    
    sim = PhantomWalk(
        initial_state=system.hoomd_snapshot,
        forcefield=system.hoomd_forcefield,
        gsd_write_freq=10,
        log_write_freq=50,
        n_steps_dpd=500,
        n_steps_fire=100,
    )
    
    sim.flush_writers()
    snap = sim.state.get_snapshot()
    system.update_positions(snap)
    return system

In [20]:
system = initialize_dense_cg_melt(number_molecules=3,chain_length=4,density=0.8)

Initializing simulation state from a gsd.hoomd.Frame.
Step 30 of 500; TPS: 306.63; ETA: 0.0 minutes
Step 60 of 500; TPS: 610.66; ETA: 0.0 minutes
Step 90 of 500; TPS: 914.25; ETA: 0.0 minutes
Step 120 of 500; TPS: 1216.2; ETA: 0.0 minutes
Step 150 of 500; TPS: 1516.79; ETA: 0.0 minutes
Step 180 of 500; TPS: 1816.88; ETA: 0.0 minutes
Step 210 of 500; TPS: 2114.82; ETA: 0.0 minutes
Step 240 of 500; TPS: 2412.69; ETA: 0.0 minutes
Step 270 of 500; TPS: 2708.23; ETA: 0.0 minutes
Step 300 of 500; TPS: 3002.82; ETA: 0.0 minutes
Step 330 of 500; TPS: 3297.49; ETA: 0.0 minutes
Step 360 of 500; TPS: 3589.63; ETA: 0.0 minutes
Step 390 of 500; TPS: 3882.22; ETA: 0.0 minutes
Step 420 of 500; TPS: 4172.17; ETA: 0.0 minutes
Step 450 of 500; TPS: 4461.41; ETA: 0.0 minutes
Step 480 of 500; TPS: 4750.97; ETA: 0.0 minutes
Step 10 of 100; TPS: 200000.0; ETA: 0.0 minutes
Step 40 of 100; TPS: 231213.87; ETA: 0.0 minutes
Step 70 of 100; TPS: 210210.21; ETA: 0.0 minutes
The closest particles after DPD + FIRE 